# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


### Finding A — Finding #3, "Click Capture by Position Tier" (paper p. 8)

**What the paper says:** weighted CTR falls sharply as position tier gets worse — Top 3: 0.423%, Page 1: 0.339%, Striking: 0.325%, Page 3-5: 0.163%, Deep: 0.050% (an 88% drop from top 3 to deep). The recommendation is to "refine search snippets on pages that already rank on page 1," reasoning that small gains there "usually convert into click lift faster than rebuilding weak pages from scratch."

**Methodology question — does the evidence carry the "faster" claim?**
The CTR-by-tier table is one static, portfolio-level snapshot: total clicks divided by total impressions per tier, computed once. It is a real, well-labeled aggregate comparison, and the paper is explicit that these are its own weighted portfolio CTRs, not generic benchmarks. But the recommendation compares the *speed* of two different actions (page-1 snippet edits vs. rebuilding weak pages), and no page was actually tracked through a snippet change and re-measured — there is no before/after pair anywhere in this finding. So the CTR curve supports "click capture compresses quickly once a page leaves the best positions" (observed, measured), but not "snippet edits pay off faster than rebuilds" (that is a comparison of two actions, and would need a before/after or a matched pair of interventions to support it directly). What would make it stronger: the same CTR-by-tier table recomputed 30–60 days after a batch of real snippet edits on a sample of page-1 URLs.


### Finding B — ML Appendix, "What Predicts Growth?" logistic regression (paper p. 29)

**What the paper says:** "Logistic regression (71% holdout accuracy) describing which sampled features separate growing from declining pages," on an 80/20 split of the 61,790-record active-content ML sample (Methodology, p. 36).

**Methodology question — where is the label from, and what does the split actually test?**
1. *Label origin.* The paper does not spell out the growth/decline label for this appendix page directly, but Finding #1 defines "up"/"down" from the 30-day-vs-previous-30-day impression change — the same kind of rule-derived proxy label we use ourselves (`trend_direction`). If the ML appendix reuses that definition, it is worth naming explicitly as a proxy for a business outcome, not an observed outcome like "clicks six months later."
2. *Split design.* The methodology section states an 80/20 split and reports no p-values or confidence intervals — the paper is upfront about that limit. What it does not say is whether pages from the same brand can sit on both sides of the split, or what the test set's base rate was. From Finding #1's growing/declining counts (74,187 vs 45,272 ≈ 62% growing) the majority-class guess in a similarly-mixed sample could already land near 60%, which would make a 71% holdout accuracy a real but fairly modest lift — not obviously visible from the single number.

**What would make it stronger:** state whether the split is row-level or brand-grouped, report the base rate next to the 71%, and show the same number again under a brand-grouped holdout — exactly the client-holdout vs. random-split comparison this notebook runs on our own model in Section 2.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


### 2.0 Setup — rebuild the Week-5 pipeline as reusable functions

Same label, same baseline formula, same feature list as `w05_model.ipynb`. Wrapping it in functions lets us run it under *several* splits without duplicating the pipeline code — the model and features stay identical; only the split changes.


In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

repo_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)
os.chdir(repo_root)

raw = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
print("Rows:", len(raw), " | Clients:", raw["client_id"].nunique())


Rows: 30000  | Clients: 32


In [2]:
def percentile_rank(s: pd.Series) -> pd.Series:
    return s.rank(method="average", pct=True).fillna(0)


def normalize(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def add_label_and_baseline(frame: pd.DataFrame) -> pd.DataFrame:
    """Same label + Week-4 baseline formula as w05_model.ipynb — frozen, unchanged."""
    frame = frame.copy()
    frame["is_declining_label"] = frame["trend_direction"].str.lower().eq("down").astype(int)

    frame["visibility_score"] = percentile_rank(np.log1p(frame["impressions_90d"]))
    frame["freshness_risk_score"] = percentile_rank(frame["days_since_last_update"])
    frame["position_opportunity_score"] = (
        (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
        * frame["visibility_score"]
        * (frame["avg_position"] > 0).astype(int)
    )
    frame["depth_gap_score"] = (1 - percentile_rank(frame["word_count"])) * frame["visibility_score"]
    frame["baseline_score"] = (
        0.40 * frame["visibility_score"]
        + 0.30 * frame["freshness_risk_score"]
        + 0.25 * frame["position_opportunity_score"]
        + 0.05 * frame["depth_gap_score"]
    ).clip(0, 1)
    return frame


# The Week-5 feature list. trend_direction / trend_pct are never in here — they define the label.
NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "days_with_impressions",
    "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]


def build_feature_matrix(frame: pd.DataFrame, numeric=NUMERIC_FEATURES, categorical=CATEGORICAL_FEATURES) -> pd.DataFrame:
    frame = frame.copy()
    for col in numeric:
        frame[col] = pd.to_numeric(frame[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    for col in categorical:
        frame[col] = frame[col].fillna("unknown").astype(str)

    numeric = list(numeric)
    if "impressions_90d" in numeric:
        frame["log_impressions_90d"] = np.log1p(frame["impressions_90d"])
        numeric.append("log_impressions_90d")
    if "clicks_90d" in numeric:
        frame["log_clicks_90d"] = np.log1p(frame["clicks_90d"])
        numeric.append("log_clicks_90d")
    if "sessions_90d" in numeric:
        frame["log_sessions_90d"] = np.log1p(frame["sessions_90d"])
        numeric.append("log_sessions_90d")

    cat_encoded = pd.get_dummies(frame[categorical], prefix=categorical, dtype=float) if categorical else pd.DataFrame(index=frame.index)
    return pd.concat([frame[numeric].reset_index(drop=True), cat_encoded.reset_index(drop=True)], axis=1)


def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order[: min(k, len(order))]]
    return float(top.mean())


df = add_label_and_baseline(raw)
print("Base rate (declining), full data:", round(df["is_declining_label"].mean(), 3))


Base rate (declining), full data: 0.542


In [3]:
def fit_and_score(df, feature_frame, train_idx, test_idx, k_list=(20, 50), seed=42):
    """Trains baseline (frozen), Logistic Regression, and Random Forest on train_idx,
    scores them on test_idx. Same model settings as w05_model.ipynb."""
    y_train = df["is_declining_label"].iloc[train_idx]
    y_test = df["is_declining_label"].iloc[test_idx]
    X_train, X_test = feature_frame.iloc[train_idx], feature_frame.iloc[test_idx]

    baseline_scores = df["baseline_score"].iloc[test_idx].to_numpy()

    logreg = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed)),
    ])
    logreg.fit(X_train, y_train)
    logreg_scores = logreg.predict_proba(X_test)[:, 1]

    rf = RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight="balanced_subsample", random_state=seed, n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    rf_scores = rf.predict_proba(X_test)[:, 1]

    row = {"n_test": len(test_idx), "base_rate": float(y_test.mean())}
    for name, scores in [("baseline", baseline_scores), ("logreg", logreg_scores), ("rf", rf_scores)]:
        for k in k_list:
            row[f"{name}_p{k}"] = precision_at_k(y_test, scores, k)
        row[f"{name}_roc_auc"] = roc_auc_score(y_test, scores)
        row[f"{name}_avg_precision"] = average_precision_score(y_test, scores)
    row["_rf_model"] = rf
    row["_rf_scores"] = rf_scores
    row["_test_idx"] = test_idx
    return row


### 2.1 "Before" — the single client-holdout split from Week 5 (unchanged)

Same seed, same 20% of *clients* held out entirely, exactly as in `w05_model.ipynb`. This reproduces last week's headline number so it can sit next to the grouped-CV result below.


In [4]:
feature_frame = build_feature_matrix(df)

client_series = df["client_id"].astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
before_train_idx = np.where(~test_mask)[0]
before_test_idx = np.where(test_mask)[0]

before_result = fit_and_score(df, feature_frame, before_train_idx, before_test_idx)

before_table = pd.DataFrame([{
    "split": "before: single client-holdout (Week 5, seed 42)",
    "n_test": before_result["n_test"],
    "test_clients": client_series.iloc[before_test_idx].nunique(),
    "base_rate": before_result["base_rate"],
    "baseline_p50": before_result["baseline_p50"],
    "logreg_p50": before_result["logreg_p50"],
    "rf_p50": before_result["rf_p50"],
    "rf_roc_auc": before_result["rf_roc_auc"],
}])
before_table


,split,n_test,test_clients,base_rate,baseline_p50,logreg_p50,rf_p50,rf_roc_auc
0,"before: single client-holdout (Week 5, seed 42)",2325,6,0.390968,0.22,0.32,0.74,0.751989


### 2.2 "After" — grouped cross-validation, every client held out exactly once

Instead of one 20%-of-clients split, `GroupKFold` rotates through **5 folds**, so every client is in the test set exactly once and never appears in both train and test within a fold. We report every fold — not just the average — because Week 5 already showed that a single client can dominate the top of the queue.


In [5]:
gkf = GroupKFold(n_splits=5)
groups = client_series.to_numpy()

fold_results = []
for fold_i, (tr_idx, te_idx) in enumerate(
    gkf.split(feature_frame, df["is_declining_label"], groups=groups), start=1
):
    result = fit_and_score(df, feature_frame, tr_idx, te_idx)
    result["fold"] = fold_i
    result["test_clients"] = client_series.iloc[te_idx].nunique()
    fold_results.append(result)

fold_table = pd.DataFrame(fold_results)[
    ["fold", "n_test", "test_clients", "base_rate",
     "baseline_p50", "logreg_p50", "rf_p50", "rf_roc_auc"]
]
fold_table


,fold,n_test,test_clients,base_rate,baseline_p50,logreg_p50,rf_p50,rf_roc_auc
0,1,7008,1,0.490154,0.28,0.76,0.90,0.651640
1,2,5731,7,0.645437,0.42,0.90,0.90,0.608342
2,3,5753,8,0.379454,0.32,0.56,0.84,0.741438
3,4,5755,8,0.622242,0.58,0.78,0.68,0.666563
4,5,5753,8,0.584738,0.68,0.86,0.44,0.667484


In [6]:
# Is the ~10-point Random-Forest-over-baseline gap a stable winner, or an average hiding a wobble?
diffs = fold_table["rf_p50"].to_numpy() - fold_table["baseline_p50"].to_numpy()
print("Per-fold (Random Forest − baseline) at Precision@50:", np.round(diffs, 3))
print("Folds where the forest wins:", int((diffs > 0).sum()), "of", len(diffs))
print("Mean difference across folds:", round(diffs.mean(), 3))

boot_rng = np.random.default_rng(42)
boot_means = np.array([
    boot_rng.choice(diffs, size=len(diffs), replace=True).mean()
    for _ in range(10_000)
])
ci_low, ci_high = np.percentile(boot_means, [5, 95])
print(f"Bootstrap 90% interval for the mean fold difference: [{ci_low:.3f}, {ci_high:.3f}]")
print("Does the interval include zero?", ci_low <= 0 <= ci_high)


Per-fold (Random Forest − baseline) at Precision@50: [ 0.62  0.48  0.52  0.1  -0.24]
Folds where the forest wins: 4 of 5
Mean difference across folds: 0.296
Bootstrap 90% interval for the mean fold difference: [0.048, 0.524]
Does the interval include zero? False


In [7]:
# Client concentration check: does one client still dominate the top of each fold's queue?
for result in fold_results:
    te_idx = result["_test_idx"]
    top50 = (
        df.iloc[te_idx]
        .assign(rf_probability=result["_rf_scores"])
        .sort_values("rf_probability", ascending=False)
        .head(50)
    )
    top_client, top_share = top50["client_id"].value_counts(normalize=True).index[0], top50["client_id"].value_counts(normalize=True).iloc[0]
    print(f"Fold {result['fold']}: top-50 queue is {top_share:.0%} one client ({result['test_clients']} clients in this fold's test set)")


Fold 1: top-50 queue is 100% one client (1 clients in this fold's test set)
Fold 2: top-50 queue is 62% one client (7 clients in this fold's test set)
Fold 3: top-50 queue is 52% one client (8 clients in this fold's test set)
Fold 4: top-50 queue is 76% one client (8 clients in this fold's test set)
Fold 5: top-50 queue is 100% one client (8 clients in this fold's test set)


### 2.3 Before vs. after, read together

*Fill this in after running the cells above — the numbers will differ slightly run to run only if a seed changes; write down what you actually see.*

Compare:
- the single Week-5 split's `rf_p50` and base rate (Section 2.1) against
- the 5-fold spread in `fold_table` and the bootstrap interval (Section 2.2).

Things worth writing a sentence about:
- Does the single-split number sit inside, above, or below the 5-fold spread?
- Does the bootstrap interval on the (Random Forest − baseline) difference include zero? If it does, the headline "Random Forest wins clearly" from Week 5 was written from one lucky (or unlucky) client draw, not from a stable pattern.
- Does the client-concentration check show the same single-client dominance in every fold, or was Week 5's `client_f74efabef1` result specific to that one draw?


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


### 3.1 The suspect: 90-day features overlap the label's own window

The label compares `impressions_last_30d` to `impressions_prev_30d`. But several features that go into the model are 90-day aggregates — `impressions_90d`, `clicks_90d`, `sessions_90d`, `days_with_impressions`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` — which by construction **already contain** that same trailing-30-day window the label is computed from. `trend_direction` and `trend_pct` are correctly never used as features (confirmed above: they are not in `NUMERIC_FEATURES`/`CATEGORICAL_FEATURES`), so this is not the label-derived-column kind of leak — it is the second kind from the `hunting-leakage-and-validating` skill: an **overlapping window**.

Test: retrain the grouped-CV model with only pre-window-safe fields (`*_prev_30d` counts plus content properties that don't touch the last-30-day window) and compare Precision@50 to the full feature set.


In [8]:
SAFE_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
]
SAFE_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
]

safe_feature_frame = build_feature_matrix(df, SAFE_NUMERIC_FEATURES, SAFE_CATEGORICAL_FEATURES)
print("Full feature set:", feature_frame.shape[1], "columns")
print("Prev-30d-only ('safe') feature set:", safe_feature_frame.shape[1], "columns")


Full feature set: 54 columns
Prev-30d-only ('safe') feature set: 35 columns


In [9]:
def grouped_precision50(df, features, groups, label):
    gkf = GroupKFold(n_splits=5)
    scores = []
    for tr_idx, te_idx in gkf.split(features, df["is_declining_label"], groups=groups):
        rf = RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1,
        )
        rf.fit(features.iloc[tr_idx], df["is_declining_label"].iloc[tr_idx])
        p = rf.predict_proba(features.iloc[te_idx])[:, 1]
        scores.append(precision_at_k(df["is_declining_label"].iloc[te_idx], p, 50))
    scores = np.array(scores)
    print(f"{label}: fold Precision@50 = {np.round(scores, 3)}  |  mean = {scores.mean():.3f}")
    return scores


print("WITH the 90-day / trailing-window features (current Week-5 feature set):")
with_suspect_scores = grouped_precision50(df, feature_frame, groups, "  full feature set")

print("\nWITHOUT them — prev-30d counts only:")
without_suspect_scores = grouped_precision50(df, safe_feature_frame, groups, "  prev-30d-only")

print(f"\nMean Precision@50 drop when the suspect features are removed: {with_suspect_scores.mean() - without_suspect_scores.mean():.3f}")


WITH the 90-day / trailing-window features (current Week-5 feature set):
  full feature set: fold Precision@50 = [0.9  0.9  0.84 0.68 0.44]  |  mean = 0.752

WITHOUT them — prev-30d counts only:
  prev-30d-only: fold Precision@50 = [0.86 0.78 0.76 0.82 0.42]  |  mean = 0.728

Mean Precision@50 drop when the suspect features are removed: 0.024


### 3.2 Sanity check — does the audit harness actually catch a real leak?

The `hunting-leakage-and-validating` skill's own advice: deliberately inject something that IS leakage and confirm the score jumps toward 1.0. If it doesn't, the test itself is broken, not the model.


In [10]:
leaky_feature_frame = feature_frame.copy()
leaky_feature_frame["trend_pct_INJECTED_LEAK"] = pd.to_numeric(df["trend_pct"], errors="coerce").fillna(0)

print("Full feature set + trend_pct injected on purpose (this SHOULD look almost perfect):")
leaky_scores = grouped_precision50(df, leaky_feature_frame, groups, "  full + injected trend_pct")


Full feature set + trend_pct injected on purpose (this SHOULD look almost perfect):
  full + injected trend_pct: fold Precision@50 = [1. 1. 1. 1. 1.]  |  mean = 1.000


### 3.3 What this means

*Fill in after running 3.1 and 3.2:*
- Did removing the 90-day/window features change Precision@50 by a little or a lot? A small drop still means the model leans partly on fields that share a window with the label — that's a design risk to name, even without a dramatic collapse. A large drop is a stronger signal.
- Did injecting `trend_pct` push Precision@50 close to 1.0? If yes, the audit harness itself works, which is what makes the honest (smaller, or unchanged) result in 3.1 trustworthy rather than a broken test.
- Product-decision flags (`health_score`, `priority_score`, etc.) are never present in this dataset in the first place, so that leakage path does not apply here — worth stating explicitly rather than silently skipping it.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


**Original claim (Week 5, Section 3):**

> "Random Forest wins on every single metric here, clearly and consistently — there is no metric where the baseline or Logistic Regression comes out ahead."

**What's wrong with it:**
- It was written from **one** client-holdout split (6 test clients, seed 42). Section 2 shows the (Random Forest − baseline) gap flips sign on at least one of five folds and the bootstrap interval on the mean gap [fill in the actual interval from Section 2.2] — so "wins on every single metric, clearly and consistently" describes one draw, not a stable pattern.
- "Wins" implies the ranking would hold for a brand-new client. Section 2.2's per-fold client-concentration check shows the top of the queue is dominated by a single client in more than one fold — so part of what looks like model skill is one client's pages looking unusually "exposed and aging."
- The lane is CTR/Engagement Opportunity Scoring, but `ctr` ranks only 10th in feature importance (Week 5, Section 4) — the model is mostly an exposure/age-decline detector, not a CTR-specific one. The original sentence doesn't say that.
- No causal claim was intended, but "wins" reads as a settled fact rather than a measured comparison.

**Rewrite:**

> Observed: under a client-grouped evaluation, Random Forest outranked the hand-written baseline at Precision@50 in [X of 5] folds. Measured: the mean gap across folds is [Y], with a bootstrap 90% interval of [Z] — [does / does not] exclude zero. Directional: the forest tends to rank stronger than the baseline across held-out clients, but the size of that advantage varies a lot by which client is held out, and the ranking is currently driven more by exposure and content age than by CTR itself. Decision-support: this supports using the forest's ranked queue as a first-pass filter for human review, not as an automatic CTR/engagement judgment, and it should be re-checked whenever a new client's pages dominate the top of the queue.

*(Fill in the bracketed numbers from your own Section 2.2 output before committing.)*


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
